In [1]:
import sys, os
sys.path.insert(1, '../..')

In [2]:
import psycopg2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from itertools import combinations
import config.config as conf

In [3]:
net_result = f'{conf.DATA_PATH}{conf.NETWORK_RESULT}'
net_src = f'{conf.DATA_PATH}{conf.NETWORK_SRC}'


In [4]:
conn = psycopg2.connect(host = conf.database_user['host'], dbname=conf.database_user['dbname'], user=conf.database_user['user'], password=conf.database_user['password'])
try:
    cur = conn.cursor()
    cur.execute("\
select \
                  x.q_id \
                , x.q_posttypeid \
                , x.q_acceptedanswerid \
                , x.q_parentid \
                , x.q_creationdate \
                , x.q_score \
                , x.q_viewcount \
                , x.q_owneruserid \
                , x.q_title \
                , x.q_tags \
                , x.q_answercount \
                , x.q_commentcount \
                , b.id               as a_id \
                , b.posttypeid       as a_posttypeid \
                , b.acceptedanswerid as a_acceptedanswerid \
                , b.parentid         as a_parentid \
                , b.creationdate     as a_creationdate \
                , b.score            as a_score \
                , b.viewcount        as a_viewcount \
                , b.owneruserid      as a_owneruserid \
                , b.title            as a_title \
                , b.tags             as a_tags \
                , b.answercount      as a_answercount \
                , b.commentcount     as a_commentcount \
  from ( \
           select a.id               as q_id \
                , a.posttypeid       as q_posttypeid \
                , a.acceptedanswerid as q_acceptedanswerid \
                , a.parentid         as q_parentid \
                , a.creationdate     as q_creationdate \
                , a.score            as q_score \
                , a.viewcount        as q_viewcount \
                , a.owneruserid      as q_owneruserid \
                , a.title            as q_title \
                , a.answercount      as q_answercount \
                , a.commentcount     as q_commentcount \
                , replace(replace(lower(a.tags), '<', ''), '>', ' ')as q_tags \
           from public_for_2324.posts a \
           where a.creationdate >= '2021-11-30' \
             and a.creationdate < '2024-12-01' \
             and a.posttypeid = '1' \
             and a.owneruserid is not null \
             and (replace(replace(lower(a.tags), '<', ''), '>', ' ') like '%python %') \
       )   x \
        , public_for_2324.posts b \
where b.parentid = x.q_id \
  and b.posttypeid = '2' \
  and b.owneruserid is not null \
; \
                " 
   )
    rows = cur.fetchall()
    

except psycopg2.DatabaseError as db_err:
    print(db_err)
finally : 
  cur.close()

In [5]:
df = pd.DataFrame(rows, columns = [
  'q_id' 
, 'q_posttypeid' 
, 'q_acceptedanswerid'
, 'q_parentid' 
, 'q_creationdate' 
, 'q_score' 
, 'q_viewcount' 
, 'q_owneruserid' 
, 'q_title' 
, 'q_tags' 
, 'q_answercount' 
, 'q_commentcount' 
, 'a_id' 
, 'a_posttypeid' 
, 'a_acceptedanswerid' 
, 'a_parentid' 
, 'a_creationdate' 
, 'a_score' 
, 'a_viewcount' 
, 'a_owneruserid' 
, 'a_title' 
, 'a_tags' 
, 'a_answercount'
, 'a_commentcount'
])

In [6]:
print(df['q_creationdate'].min())
print(df['a_creationdate'].max())

2021-11-30 00:00:11.787000
2025-06-30 19:45:59.467000


In [7]:
# 답변이 1년 이내의 달린것으로 조건 수정
df = df[df['a_creationdate'] - df['q_creationdate'] <= pd.Timedelta(days=365)]

In [8]:
df_2122 = df[(df['q_creationdate']>= '2021-11-30') & (df['q_creationdate']< '2022-11-30')  ]
df_2223 = df[(df['q_creationdate']>= '2022-11-30') & (df['q_creationdate']< '2023-11-30')  ]
df_2324 = df[(df['q_creationdate']>= '2023-11-30') & (df['q_creationdate']< '2024-11-30')  ]

In [ ]:
def save_file(df, f_nm) :
    df_gephi = df[['a_owneruserid', 'q_owneruserid']]
    df_gephi.columns = ['Source', 'Target']
    print(f'{net_src}/{f_nm}.csv')
    df_gephi.to_csv(f'{net_src}/{f_nm}.csv')

In [10]:
save_file(df_2122, 'qna_2122')
save_file(df_2223, 'qna_2223')
save_file(df_2324, 'qna_2324')

/usr/share/d_ollama/data/network_src/qna_2122.csv
/usr/share/d_ollama/data/network_src/qna_2223.csv
/usr/share/d_ollama/data/network_src/qna_2324.csv


In [27]:
reputation_usrid_list = list(set(df_2122[['q_owneruserid', 'a_owneruserid']].values.flatten()))

In [28]:
conn = psycopg2.connect(host = conf.database_user['host'], dbname=conf.database_user['dbname'], user=conf.database_user['user'], password=conf.database_user['password'])

In [ ]:
reputation_dict = dict()
for usrid in reputation_usrid_list:
    sql=f"""select fn_calc_reputation('{usrid}', '2022-11-30')"""
    try:
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        

    except psycopg2.DatabaseError as db_err:
        print(db_err)
    finally : 
        cur.close()
    reputation_dict[usrid] = 0 if rows[0][0]== None else rows[0][0]

In [53]:
import json

In [62]:

reputation_dict_save = {str(k): v for k, v in reputation_dict.items()}

In [63]:
reputation_dict_save

{'16252930': 12,
 '17825796': 0,
 '10485767': 0,
 '6553609': 53,
 '18087952': 0,
 '17825811': 36,
 '19398677': 38,
 '10223638': 10,
 '5767191': 601,
 '12320789': 10,
 '4456473': 118,
 '16777241': 136,
 '18612251': -4,
 '6553631': 545,
 '10223648': 10,
 '6553634': 0,
 '11534372': -2,
 '20185124': 53,
 '8912934': 127,
 '19922984': 0,
 '17825832': 18,
 '12058672': 259,
 '2097207': 2967,
 '4194360': 62,
 '13369404': 40,
 '17563711': 66,
 '13107269': 0,
 '16777287': 0,
 '11534407': 35,
 '16253004': 32,
 '524368': 156051,
 '2097233': 150,
 '3670097': 888,
 '10485842': 30,
 '19660884': 10,
 '19660885': 0,
 '11272278': 10,
 '9961559': 85,
 '2097240': 77439,
 '18088024': -2,
 '7864411': 26,
 '17563740': 4,
 '14155869': 13,
 '17825886': 20,
 '95': 14842,
 '9175134': 66,
 '20447329': 0,
 '7340130': 227,
 '17825891': 10,
 '17825883': 0,
 '17563743': -6,
 '13631591': 660,
 '19923056': 0,
 '8388721': 40,
 '20447347': 0,
 '17825908': 10,
 '4194419': 735,
 '786559': 3471,
 '13107330': 0,
 '8913028': 0

In [66]:
with open(f'{net_src}/reputation_dict.json', 'w', encoding='utf-8') as f:
    json.dump(reputation_dict_save, f, indent=4, ensure_ascii=False)